In [40]:
import json
import os
import pandas as pd
import oracledb
from pyproj import Transformer

# If you don't have oracledb installed, install via:
# %pip install oracledb
pd.set_option("display.max_rows", 500) 

oracledb.init_oracle_client(lib_dir=r"C:\Oracle\instantclient_23_4")
print("Thin mode = ", oracledb.is_thin_mode())   # should print False

config_path = r"C:\Workspace\GIT\USACE-AIS-Scripts\config.json"

with open(config_path, "r") as f:
    config = json.load(f)

# Pull GIS database parameters from config
dsn = config["gis_server"] 
db_user = config["gis_db_username"]
db_pass = config["gis_db_password"]

# --------------------------------------------------
# 2. Connect to Oracle
# --------------------------------------------------
# For thick mode (using Oracle Instant Client), you might need:
# oracledb.init_oracle_client(lib_dir=r"C:\path\to\instantclient")

connection = oracledb.connect(
    user=db_user,
    password=db_pass,
    dsn=dsn
)

Thin mode =  False


In [44]:
sql = """
WITH proj_states AS (
    SELECT
        PROJECT_ID,
        LISTAGG(STATE_NAME, ', ') WITHIN GROUP (ORDER BY STATE_NAME) AS STATE_NAMES
    FROM UMRRDB.PROJECT_STATES
    GROUP BY PROJECT_ID
),
proj_counties AS (
    SELECT
        PROJECT_ID,
        LISTAGG(STATE_COUNTY, ', ') WITHIN GROUP (ORDER BY STATE_COUNTY) AS STATE_COUNTIES
    FROM UMRRDB.PROJECT_COUNTIES
    GROUP BY PROJECT_ID
)
SELECT
    p.*,
    ps.STATE_NAMES,
    pc.STATE_COUNTIES,
    sdo_geom.sdo_centroid(p.SHAPE, 0.005).sdo_point.x AS CENTROID_X,
    sdo_geom.sdo_centroid(p.SHAPE, 0.005).sdo_point.y AS CENTROID_Y
FROM UMRRDB.PROJECTS p
LEFT JOIN proj_states   ps ON p.PROJECT_ID = ps.PROJECT_ID
LEFT JOIN proj_counties pc ON p.PROJECT_ID = pc.PROJECT_ID
"""

print("Getting projects...")
df_projects = pd.read_sql(sql, con=connection)

print("Transforming coordinates...")
# Source CRS: NAD83 / UTM zone 15N (EPSG:26915)
transformer = Transformer.from_crs("EPSG:26915", "EPSG:4326", always_xy=True)

def to_wgs84(x, y):
    if x is None or y is None:
        return (None, None)
    lon, lat = transformer.transform(x, y)
    return lon, lat

df_projects["CENTROID_LON"], df_projects["CENTROID_LAT"] = zip(
    *df_projects.apply(lambda r: to_wgs84(r["CENTROID_X"], r["CENTROID_Y"]), axis=1)
)

df_projects = df_projects[df_projects["DISTRICT"] == "MVR"]
df_projects

Getting projects...


C:\Users\b5edgr9b\AppData\Local\Temp\1\ipykernel_45724\1730486213.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_projects = pd.read_sql(sql, con=connection)


Transforming coordinates...


,OBJECTID,PROJECT_ID,PROJECT_NAME,PROJECT_NAME_SHORT,DISTRICT,POOL,PHASE,STATUS,MAP_NUMBER,PERCENT_COMPLETE,...,BENEFIT_SOURCE,SHAPE,SE_ANNO_CAD_DATA,STORYMAP_URL,STATE_NAMES,STATE_COUNTIES,CENTROID_X,CENTROID_Y,CENTROID_LON,CENTROID_LAT
37,16,61,Chautauqua Refuge Habitat and Rehabilitation a...,Chautauqua Refuge,MVR,La Grange Pool,Post Construction,Complete,17.0,100.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...",Illinois,IL - Mason,754558.246250,4.473778e+06,-90.001539,40.375731
38,22,62,Banner Marsh Habitat Rehabilitation and Enhanc...,Banner Marsh,MVR,La Grange Pool,Post Construction,Complete,4.0,100.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...",Illinois,"IL - Fulton, IL - Peoria",766063.010882,4.491219e+06,-89.858908,40.529034
39,95,63,"Rice Lake, IL Habitat Rehabilitation and Enhan...","Rice Lake, IL",MVR,La Grange Pool,Post Construction,Complete,70.0,99.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...",Illinois,IL - Fulton,760768.384506,4.484033e+06,-89.924286,40.466065
40,6,64,Peoria Lake Enhancement,Peoria Lake,MVR,Peoria Pool,Post Construction,Complete,52.0,100.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...",Illinois,"IL - Peoria, IL - Woodford",797453.837298,4.533224e+06,-89.469022,40.896066
41,73,65,Bertom McCartney Lakes Habitat Rehabilitation ...,Bertom McCartney Lakes,MVR,Pool 11,Post Construction,Complete,9.0,100.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...",Wisconsin,WI - Grant,672083.394979,4.728088e+06,-90.899466,42.686018
42,35,67,Turkey River Bottoms Delta and Backwater Compl...,Turkey River Bottoms Delta and Backwater Complex,MVR,Pool 11,Feasibility,Deferred,84.0,1.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...","Iowa, Wisconsin","IA - Clayton, WI - Grant",664676.694557,4.729622e+06,-90.989375,42.701451
43,74,68,Snyder Slough Backwater Complex Habitat Rehabi...,Snyder Slough Backwater Complex,MVR,Pool 11,Fact Sheet,Inactive,74.0,1.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...",Wisconsin,WI - Grant,681431.962022,4.726177e+06,-90.786045,42.666677
44,8,69,Pool 12 Overwintering Habitat Rehabilitation a...,"Pool 12 Overwintering, IL",MVR,Pool 12,Post Construction,Complete,59.0,99.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...",Illinois,IL - Jo Daviess,706700.979761,4.696466e+06,-90.488705,42.392993
45,75,70,Brown's Lake Habitat Rehabilitation and Enhanc...,Brown's Lake,MVR,Pool 13,Post Construction,Complete,13.0,100.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...",Iowa,IA - Jackson,726518.485016,4.670756e+06,-90.258242,42.156185
46,76,71,Potters Marsh Habitat Rehabilitation and Enhan...,Potters Marsh,MVR,Pool 13,Post Construction,Complete,67.0,100.0,...,NaN,<oracledb.DbObject MDSYS.SDO_GEOMETRY at 0x1f7...,None,"<a href=""https://www.mvr.usace.army.mil/Missio...",Illinois,"IL - Carroll, IL - Whiteside",738745.032773,4.646255e+06,-90.120408,41.932158


In [45]:
df_projects.to_csv(r"C:\Workspace\GIT\USACE-AIS-Scripts\UMRR\UMRR_Projects.csv", index=False)

In [ ]:
connection.close()